In [ ]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import h5py
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from more_itertools import one
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints")

jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [ ]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [ ]:
dataset_test = xr.open_dataset(
    datadir
    / "datasets/shallow_water_1d/data/test/shallow_water_1d_test_grf_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type in ("NaiveFluxNO",):  # ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=1500,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
import json


with open("../../data/analysis/shallow_water_indistribution.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

## Long term

In [ ]:
@eqx.filter_jit
def metrics_long_rollout(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]
    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]
    l2 = jax.vmap(relative_L2_error)(u_pred, u_data)
    l_inf = jax.vmap(relative_L_infty_error)(u_pred, u_data)
    return {"l2": l2, "l_inf": l_inf}

In [ ]:
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: metrics_long_rollout(model, u, (dt, dx))),
            values,
            batch_size=1000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(results)

In [ ]:
results_stacked = {
    k: jax.tree.transpose(
        jax.tree.structure(["*"] * 3),
        None,
        v,
    )
    for k, v in results_dict.items()
}
results_stacked = jax.tree.map(
    np.stack, results_stacked, is_leaf=lambda x: isinstance(x, list)
)

In [ ]:
with h5py.File("../../data/analysis/shallow_water_1d_long_rollout.hdf5", "w") as f:
    for model_name, res_dict in results_stacked.items():
        g = f.create_group(model_name)
        for metric_name, val in res_dict.items():
            g.create_dataset(metric_name, data=val)

## OOD

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "shallow_water_1d_test_step_grf_seed=10.hdf5",
    # / "datasets/shallow_water_1d/data/test/shallow_water_1d_test_step_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
import matplotlib.pyplot as plt


fig, axes = plt.subplots(2, 5, figsize=(10, 5), sharey="row")
for i in range(axes.shape[1]):
    axes[0, i].plot(dataset_test.coords["x"], dataset_test["values"][0, 4 * i, 0])
    axes[1, i].plot(dataset_test.coords["x"], dataset_test["values"][0, 4 * i, 1])

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
# for model_type in (
#     "DISCO",
#     "DPOT",
#     "HyperFluxFNOLocal",
# ):
for model_type in ("HyperNeuralOperator",):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=1000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
import json


with open("../../data/analysis/shallow_water_ood_step_grf.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    print(
        model_name,
        jax.tree.map(
            lambda list_: {
                "mean": jnp.mean(jnp.asarray(list_)),
                "std": jnp.std(jnp.asarray(list_)),
            },
            res_,
            is_leaf=lambda x: isinstance(x, list),
        ),
    )